# JARVIS — Kaggle One-Click Launcher

This notebook launches the complete JARVIS tactical assistant: cinematic React HUD, voice-first conversation, multilingual transcription, camera object detection, live location/weather/news context, creator identity, and the Custom API provider panel.

**Run every cell from top to bottom.** The notebook is self-contained and includes a source fallback, so it can run even when no Dataset is attached. If a JARVIS source ZIP is attached in `/kaggle/input`, that ZIP is used instead.

Optional Kaggle Secrets: `MANUS_API_URL` and `MANUS_API_KEY` enable managed AI, transcription, and vision routes. You can also open **Custom API** inside the site and provide any OpenAI-compatible provider without adding a Kaggle Secret.

In [ ]:
from pathlib import Path
import atexit, base64, json, os, shutil, socket, subprocess, sys, time, urllib.error, urllib.request, zipfile

WORK = Path(os.environ.get("JARVIS_KAGGLE_WORKDIR", "/kaggle/working"))
PROJECT = WORK / "jarvis"
PORT = 3000
SERVER_LOG = WORK / "jarvis-server.log"
server = None

# Embedded source fallback: lets the notebook run without a separate Dataset.
EMBEDDED_SOURCE_B64 = """..."""

def run(command, cwd=None, env=None, label=None):
    print(f"\n>>> {label or ' '.join(command)}")
    result = subprocess.run(command, cwd=cwd, env=env, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(command)}")
    return result

def find_project(root):
    root = Path(root)
    if (root / "package.json").exists():
        return root
    candidates = sorted(root.rglob("package.json"), key=lambda p: (len(p.parts), str(p)))
    return candidates[0].parent if candidates else None

def extract_zip(zip_path, destination):
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)
    return find_project(destination)

def prepare_source():
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    PROJECT.mkdir(parents=True)

    input_root = Path(os.environ.get("JARVIS_KAGGLE_INPUT", "/kaggle/input"))
    candidates = []
    if input_root.exists():
        candidates = sorted(input_root.rglob("*.zip"), key=lambda p: (0 if "jarvis" in p.name.lower() else 1, str(p)))
    if candidates:
        extracted = WORK / "jarvis-input-source"
        shutil.rmtree(extracted, ignore_errors=True)
        extracted.mkdir(parents=True)
        source = extract_zip(candidates[0], extracted)
        print("Using attached source ZIP:", candidates[0])
    else:
        embedded_zip = WORK / "jarvis-embedded-source.zip"
        embedded_zip.write_bytes(base64.b64decode(EMBEDDED_SOURCE_B64))
        source = extract_zip(embedded_zip, WORK / "jarvis-embedded-source")
        print("Using the notebook's embedded source bundle.")
    if source is None or not (source / "package.json").exists():
        raise FileNotFoundError("No valid JARVIS source was found.")
    shutil.copytree(source, PROJECT, dirs_exist_ok=True)
    print("Project prepared at:", PROJECT)
    print("Package:", json.loads((PROJECT / "package.json").read_text())["name"])

prepare_source()

In [ ]:
# Install the pinned package manager and dependencies.
run(["npm", "install", "--global", "pnpm@10.18.0"], label="Install pnpm 10.18.0")
run(["pnpm", "install", "--frozen-lockfile"], cwd=PROJECT, label="Install locked dependencies")
print("Dependencies installed successfully.")

In [ ]:
# Load optional Kaggle Secrets without ever printing their values.
try:
    from kaggle_secrets import UserSecretsClient
    secret_client = UserSecretsClient()
    for name in ("MANUS_API_URL", "MANUS_API_KEY"):
        try:
            value = secret_client.get_secret(name)
            if value:
                os.environ[name] = value
        except Exception:
            pass
except Exception:
    pass
managed_ready = bool(os.environ.get("MANUS_API_URL") and os.environ.get("MANUS_API_KEY"))
print("Managed service secrets available:", managed_ready)
print("Custom API can also be configured inside the running HUD without storing a key in Kaggle.")

In [ ]:
# Validate the complete source and build the production bundle.
run(["pnpm", "check"], cwd=PROJECT, label="TypeScript validation")
run(["pnpm", "build"], cwd=PROJECT, label="Production build")
assert (PROJECT / "dist" / "index.js").exists(), "Production server bundle was not created."
assert (PROJECT / "dist" / "public" / "index.html").exists(), "Frontend bundle was not created."
assert (PROJECT / "public" / "manus-routes.json").exists(), "Route manifest is missing."
print("JARVIS validation and production build completed.")

In [ ]:
# Start the production server and wait until it is healthy.
def port_open(port):
    with socket.socket() as sock:
        sock.settimeout(0.2)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def stop_server():
    global server
    if server is not None and server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server.kill()

# Clean up a stale process started by an earlier notebook execution when possible.
if port_open(PORT):
    print(f"Port {PORT} is already in use; reusing the existing healthy server if possible.")

env = os.environ.copy()
env.update({"NODE_ENV": "production", "PORT": str(PORT)})
if not port_open(PORT):
    log = SERVER_LOG.open("w")
    server = subprocess.Popen(["pnpm", "start"], cwd=PROJECT, env=env, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(45):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/health", timeout=2) as response:
                if response.status == 200:
                    break
        except Exception:
            time.sleep(1)
    else:
        print(SERVER_LOG.read_text()[-6000:])
        raise RuntimeError("JARVIS server did not become ready.")
else:
    print("An existing server is already listening; continuing with it.")

atexit.register(stop_server)
print(f"JARVIS is ready at http://127.0.0.1:{PORT}")

In [ ]:
# Deterministic smoke tests for the routes needed by the site.
def get_json(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=10) as response:
        return response.status, json.loads(response.read().decode())

status, health = get_json("/api/health")
assert status == 200 and health.get("status") == "ok", health
status, routes = get_json("/manus-routes.json")
assert status == 200 and routes.get("routes"), routes

request = urllib.request.Request(f"http://127.0.0.1:{PORT}/api/custom-ai/respond", data=b"{}", headers={"Content-Type":"application/json"}, method="POST")
try:
    urllib.request.urlopen(request, timeout=10)
    raise AssertionError("Custom API validation should reject an empty request")
except urllib.error.HTTPError as error:
    assert error.code == 400, error.code

print("Health route: PASS")
print("Route manifest: PASS")
print("Custom API validation: PASS")
print("All deterministic notebook smoke tests passed.")

In [ ]:
# Display a clickable local preview link in the notebook output.
preview_url = f"http://127.0.0.1:{PORT}"
try:
    from IPython.display import HTML, display
    display(HTML(f'''<div style="padding:16px;border:1px solid #58d9f4;background:#07151b;color:#d9faff;font-family:monospace"><strong>JARVIS IS RUNNING</strong><br><br><a href="{preview_url}" target="_blank" style="color:#78eaff">Open the tactical HUD</a><br><small>Allow microphone, camera, and precise location permissions in the opened browser view.</small></div>'''))
except ImportError:
    print("JARVIS is running at", preview_url)

## Using the site

Open the link from the previous cell. Allow microphone, camera, and precise location access when prompted. The left rail includes **Custom API**, where you can enter an OpenAI-compatible base URL, model, and key or upload a `.env`/JSON configuration. Credentials remain in the current browser session only.

Kaggle sessions are temporary. To stop the server later, run `stop_server()` in a new cell.